# MVP: семантическая карта региональной медиаповестки

Этот notebook поддерживает два формальных режима работы:

1. `Работа с готовыми локальными артефактами`
2. `Полный локальный прогон`

Канонический pipeline:

1. очистка корпуса
2. построение набора candidate-статей
3. построение или переиспользование story embeddings
4. оценка экономической релевантности, присвоение тем BoE и построение регионального индекса
5. построение итоговых таблиц и графиков

Публичная версия notebook очищена от outputs с прямыми текстовыми фрагментами корпуса и предназначена для безопасного воспроизведения pipeline и агрегированных результатов.


In [ ]:
from pathlib import Path
import json
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif (PROJECT_ROOT / 'final_project').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'final_project'

CONFIG_PATH = PROJECT_ROOT / 'config' / 'settings.yaml'
TOPIC_PATH = PROJECT_ROOT / 'config' / 'topic_profiles.yaml'

with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    settings = yaml.safe_load(handle)
with TOPIC_PATH.open('r', encoding='utf-8') as handle:
    topic_profiles = yaml.safe_load(handle)['topics']

BOE_TOPICS = [topic['name'] for topic in topic_profiles]
paths = {key: PROJECT_ROOT / value for key, value in settings['paths'].items()}
PYTHON = str((PROJECT_ROOT.parent / '.venv' / 'Scripts' / 'python.exe').resolve())

RUN_CLEANING = False
RUN_CANDIDATES = False
USE_EXISTING_EMBEDDINGS = True
MODEL_SIZE = settings['models']['default_sentence_transformer_size']
RUN_MODE = settings['runtime']['run_mode']

def read_parquet_head(path, columns=None, n_rows=5000):
    parquet_file = pq.ParquetFile(path)
    first_batch = next(parquet_file.iter_batches(batch_size=n_rows, columns=columns))
    return first_batch.to_pandas()

def path_status(path):
    return 'exists' if Path(path).exists() else 'missing'

PROJECT_ROOT


In [ ]:
display(Markdown('## settings.yaml'))
display(Markdown(f'**run_mode:** `{settings["runtime"]["run_mode"]}`'))
print(CONFIG_PATH.read_text(encoding='utf-8'))


In [ ]:
display(Markdown('## topic_profiles.yaml'))
print(TOPIC_PATH.read_text(encoding='utf-8'))


In [ ]:
display(Markdown('## Режим выполнения'))
mode_label = 'Работа с готовыми локальными артефактами' if USE_EXISTING_EMBEDDINGS else 'Полный локальный прогон'
display(Markdown(f'**Выбранный режим:** `{mode_label}`'))
display(Markdown(
    f'`RUN_CLEANING={RUN_CLEANING}`, `RUN_CANDIDATES={RUN_CANDIDATES}`, '
    f'`USE_EXISTING_EMBEDDINGS={USE_EXISTING_EMBEDDINGS}`, '
    f'`MODEL_SIZE={MODEL_SIZE}`, `RUN_MODE={RUN_MODE}`'
))
display(Markdown(
    f'Текущие артефакты: `cleaned={path_status(paths["cleaned_articles"])}`, '
    f'`candidate={path_status(paths["candidate_articles"])}`, '
    f'`lookup={path_status(paths["story_lookup"])}`, '
    f'`embeddings={path_status(paths["story_embeddings"])}`'
))


## 1. Очистка и проверка корпуса


In [ ]:
if RUN_CLEANING:
    clean_run = subprocess.run(
        [PYTHON, 'scripts/01_clean_articles.py', '--config', str(CONFIG_PATH)],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    print(clean_run.stdout)
    if clean_run.stderr.strip():
        print(clean_run.stderr)
else:
    print(f'Skipping step 1. Using existing file: {paths["cleaned_articles"]}')


In [ ]:
with paths['cleaning_report'].open('r', encoding='utf-8') as handle:
    cleaning_report = json.load(handle)

summary = cleaning_report['summary']

cleaning_summary = pd.Series({
    'rows_seen': summary['rows_seen'],
    'rows_kept': summary['rows_kept'],
    'rows_dropped_short_text': summary['rows_dropped_short_text'],
    'rows_dropped_date_outlier': summary['rows_dropped_date_outlier'],
    'date_min': summary['clean_date_min'],
    'date_max': summary['clean_date_max'],
    'duplicate_story_groups': summary['duplicate_story_groups'],
}).to_frame('value')

display(cleaning_summary)

cleaned_head = read_parquet_head(
    paths['cleaned_articles'],
    columns=['unique_article_id', 'date', 'week', 'region', 'story_key', 'dup_weight', 'text_len', 'text_clean'],
    n_rows=5000,
)

display(cleaned_head.head(10))
display(cleaned_head['region'].value_counts().head(10).to_frame('rows'))
display(cleaned_head['text_len'].describe().to_frame('text_len'))


## 2. Построение candidate-набора


In [ ]:
if RUN_CANDIDATES:
    candidate_run = subprocess.run(
        [PYTHON, 'scripts/02_filter_candidates.py', '--config', str(CONFIG_PATH)],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    print(candidate_run.stdout)
    if candidate_run.stderr.strip():
        print(candidate_run.stderr)
else:
    print(f'Skipping step 2. Using existing file: {paths["candidate_articles"]}')


In [ ]:
candidate_head = read_parquet_head(
    paths['candidate_articles'],
    columns=['unique_article_id', 'date', 'week', 'region', 'candidate_source', 'strong_keyword_hit', 'weak_keyword_hit', 'url_section_hit'],
    n_rows=5000,
)
candidate_source_distribution = candidate_head['candidate_source'].value_counts(dropna=False).rename_axis('candidate_source').to_frame('rows')

candidate_summary = pd.Series({
    'candidate_rows_in_preview': len(candidate_head),
    'share_strong_keyword': float(candidate_head['strong_keyword_hit'].mean()),
    'share_weak_keyword': float(candidate_head['weak_keyword_hit'].mean()),
    'share_url_section': float(candidate_head['url_section_hit'].mean()),
    'unique_regions_in_preview': int(candidate_head['region'].nunique()),
    'unique_weeks_in_preview': int(pd.to_datetime(candidate_head['week']).nunique()),
})

display(Markdown('### Сводка candidate gate'))
display(candidate_summary.to_frame('value'))
display(Markdown('### Структура candidate_source в preview'))
display(candidate_source_distribution)
display(Markdown('### Preview candidate rows без текстовых фрагментов'))
display(candidate_head.head(20))


## 3. Построение или переиспользование story embeddings


In [ ]:
if USE_EXISTING_EMBEDDINGS:
    embedding_cmd = [
        PYTHON,
        'scripts/03_build_embeddings.py',
        '--config',
        str(CONFIG_PATH),
        '--mode',
        'bootstrap-existing',
    ]
    step3_label = 'Работа с готовыми локальными артефактами'
else:
    embedding_cmd = [
        PYTHON,
        'scripts/03_build_embeddings.py',
        '--config',
        str(CONFIG_PATH),
        '--mode',
        'build',
        '--model-size',
        MODEL_SIZE,
        '--run-mode',
        RUN_MODE,
    ]
    step3_label = 'Полный локальный прогон'

embedding_run = subprocess.run(
    embedding_cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

display(Markdown(f'### Сводка step 3 (`{step3_label}`)'))
print(embedding_run.stdout)
if embedding_run.stderr.strip():
    print(embedding_run.stderr)


In [ ]:
story_lookup = pd.read_parquet(paths['story_lookup'])
story_embeddings = np.load(paths['story_embeddings'], mmap_mode='r')

embedding_summary = pd.Series({
    'story_lookup_rows': len(story_lookup),
    'embedding_rows': int(story_embeddings.shape[0]),
    'embedding_dim': int(story_embeddings.shape[1]),
    'story_embeddings_file_mb': round(paths['story_embeddings'].stat().st_size / (1024 ** 2), 2),
    'run_mode_in_lookup': ', '.join(sorted(story_lookup['run_mode'].astype(str).unique())),
}).to_frame('value')

display(embedding_summary)
display(story_lookup[['story_key', 'region', 'week', 'region_spread', 'run_mode']].head(15))
display(story_lookup['region'].value_counts().head(10).to_frame('stories'))


## 4. Economic gate, темы BoE и региональный индекс


In [ ]:
score_cmd = [
    PYTHON,
    'scripts/04_score_and_build_index.py',
    '--config',
    str(CONFIG_PATH),
    '--model-size',
    MODEL_SIZE,
]
if USE_EXISTING_EMBEDDINGS:
    score_cmd.append('--reuse-embeddings')

score_run = subprocess.run(
    score_cmd,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)

display(Markdown(
    f'### Сводка step 4 (`model_size={MODEL_SIZE}`, `reuse_embeddings={USE_EXISTING_EMBEDDINGS}`)'
))
print(score_run.stdout)
if score_run.stderr.strip():
    print(score_run.stderr)


In [ ]:
scored_articles = pd.read_parquet(paths['scored_articles'])
regional_index = pd.read_parquet(paths['regional_topic_index'])

scored_articles['week'] = pd.to_datetime(scored_articles['week'])
regional_index['week'] = pd.to_datetime(regional_index['week'])

econ_mask = scored_articles['is_econ']
scoring_summary = pd.Series({
    'scored_articles': len(scored_articles),
    'economic_articles': int(econ_mask.sum()),
    'boe_topic_articles': int(scored_articles['assigned_topic'].isin(BOE_TOPICS).sum()),
    'other_econ_articles': int(scored_articles['assigned_topic'].eq('other_econ').sum()),
    'regions_in_index': int(regional_index['region'].nunique()),
    'weeks_in_index': int(regional_index['week'].nunique()),
    'other_econ_share_among_econ_articles': float(scored_articles.loc[econ_mask, 'assigned_topic'].eq('other_econ').mean()),
})
topic_distribution = scored_articles.loc[econ_mask, 'assigned_topic'].value_counts().rename_axis('assigned_topic').to_frame('rows')

display(Markdown('### Сводка semantic scoring'))
display(scoring_summary.to_frame('value'))
display(Markdown('### Распределение assigned_topic среди экономических статей'))
display(topic_distribution.head(12))


In [ ]:
display(Markdown('### Top region-topic pairs'))
display(regional_index.sort_values('risk_score', ascending=False).head(20))


## 5. Итоговые таблицы и графики


In [ ]:
output_run = subprocess.run(
    [PYTHON, 'scripts/05_make_outputs.py', '--config', str(CONFIG_PATH)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(output_run.stdout)
if output_run.stderr.strip():
    print(output_run.stderr)


In [ ]:
hot_pairs = pd.read_csv(paths['hot_regions_topics'])

reference_week = 'no active BoE week found'
if not hot_pairs.empty:
    reference_week = pd.to_datetime(hot_pairs['week']).max().date().isoformat()

display(Markdown(f'### Reference week used for latest outputs: `{reference_week}`'))
display(hot_pairs.head(20))
display(Markdown('Публичная версия notebook не выводит representative snippets и review tables с прямыми текстовыми фрагментами корпуса.'))


In [ ]:
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    latest_settings = yaml.safe_load(handle)

latest_paths = {key: PROJECT_ROOT / value for key, value in latest_settings['paths'].items()}

figure_specs = [
    ('### Карта: reference week', 'latest_heatmap'),
    ('### Карта: среднее за декабрь 2022', 'monthly_heatmap'),
    ('### Карта: общий top-risk по регионам', 'overall_risk_map'),
    ('### Карта: доминирующая тема по регионам', 'dominant_topic_map'),
    ('### График трендов: showcase topic', 'hot_topic_trends'),
    ('### График трендов: совокупный региональный риск', 'overall_region_trends'),
]

for title, key in figure_specs:
    path = latest_paths.get(key)
    display(Markdown(title))
    if path is None:
        display(Markdown(f'Path `{key}` is missing from `settings.yaml`.'))
        continue
    if not path.exists():
        display(Markdown(f'File not found: `{path}`'))
        continue
    display(Image(filename=str(path)))
